In [1]:

!pip install -q --upgrade pip
!pip install -q albumentations
!pip install -q torchmetrics
!pip install -q pycocotools
!pip install -q opencv-python-headless

print("✅ All required libraries have been successfully installed/updated!")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 25.3 MB/s eta 0:00:0000:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which i

In [2]:
import os
import random
import cv2
import torch
import torchvision
import numpy as np
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
from pycocotools.coco import COCO
from torch.utils.data import Dataset

def get_train_transforms():
    return A.Compose([
        A.HorizontalFlip(p=0.5),
        A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=15, 
                           border_mode=cv2.BORDER_CONSTANT, value=0, p=0.5),
        A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.5),
        ToTensorV2()
    ], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['labels']))

def get_val_transforms():
    return A.Compose([
        ToTensorV2()
    ], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['labels']))

class TeaBoxDataset(Dataset):
    def __init__(self, root_dir, annotation_file, transforms=None):
        self.root_dir = root_dir
        self.coco = COCO(annotation_file)
        self.ids = list(sorted(self.coco.imgs.keys()))
        self.transforms = transforms

    def __getitem__(self, index):
        img_id = self.ids[index]
        ann_ids = self.coco.getAnnIds(imgIds=img_id)
        anns = self.coco.loadAnns(ann_ids)
        
        path = self.coco.loadImgs(img_id)[0]['file_name']
        img_path = os.path.join(self.root_dir, path)
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        masks = []
        boxes = []
        labels = []
        
        for ann in anns:
            masks.append(self.coco.annToMask(ann))
            xmin, ymin, w, h = ann['bbox']
            boxes.append([xmin, ymin, xmin + w, ymin + h])
            labels.append(ann['category_id'])

        if self.transforms is not None and len(boxes) > 0:
            transformed = self.transforms(image=image, bboxes=boxes, masks=masks, labels=labels)
            image = transformed['image'] / 255.0
            boxes = transformed['bboxes']
            masks = transformed['masks']
            labels = transformed['labels']
        else:
            image = torch.from_numpy(image.transpose((2, 0, 1))).float() / 255.0

        target = {}
        target["boxes"] = torch.as_tensor(boxes, dtype=torch.float32)
        target["labels"] = torch.as_tensor(labels, dtype=torch.int64)
        target["masks"] = torch.as_tensor(np.array(masks), dtype=torch.uint8)
        target["image_id"] = torch.tensor([img_id])

        return image, target

    def __len__(self):
        return len(self.ids)

def collate_fn(batch):
    return tuple(zip(*batch))

BATCH_SIZE = 2
LEARNING_RATE = 0.001
NUM_CLASSES = 2 

def get_mask_rcnn_model(num_classes):
    # Use the official ResNet50 Mask R-CNN
    model = torchvision.models.detection.maskrcnn_resnet50_fpn(weights="DEFAULT")
    
    # CRITICAL: Freeze the entire backbone to prevent early overfitting and save GPU memory
    for param in model.backbone.parameters():
        param.requires_grad = False
        
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    
    in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
    hidden_layer = 256
    model.roi_heads.mask_predictor = MaskRCNNPredictor(in_features_mask, hidden_layer, num_classes)
    
    return model

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model = get_mask_rcnn_model(NUM_CLASSES)
model.to(device)

optimizer = torch.optim.SGD([p for p in model.parameters() if p.requires_grad], 
                            lr=LEARNING_RATE, momentum=0.9, weight_decay=0.001)
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

Downloading: "https://download.pytorch.org/models/maskrcnn_resnet50_fpn_coco-bf2d0c1e.pth" to /root/.cache/torch/hub/checkpoints/maskrcnn_resnet50_fpn_coco-bf2d0c1e.pth


100%|██████████| 170M/170M [00:00<00:00, 223MB/s] 


In [4]:
import gc
import torch
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from torch.utils.data import DataLoader
from tqdm import tqdm

TRAIN_IMG_DIR = "/kaggle/input/datasets/muhammadunssrahim/teabox2/train"
TRAIN_JSON = "/kaggle/input/datasets/muhammadunssrahim/teabox2/train/_annotations.coco.json"
VAL_IMG_DIR = "/kaggle/input/datasets/muhammadunssrahim/teabox2/valid"
VAL_JSON = "/kaggle/input/datasets/muhammadunssrahim/teabox2/valid/_annotations.coco.json"

NUM_EPOCHS = 15 

train_dataset = TeaBoxDataset(TRAIN_IMG_DIR, TRAIN_JSON, transforms=get_train_transforms())
val_dataset = TeaBoxDataset(VAL_IMG_DIR, VAL_JSON, transforms=get_val_transforms())

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, collate_fn=collate_fn)

print(f"\nStarting memory-optimized training pipeline for 'TeaBox found!' for {NUM_EPOCHS} epochs...")
print("=" * 80)

for epoch in range(1, NUM_EPOCHS + 1):
    
    model.train()
    total_loss = 0
    
    train_loop = tqdm(train_loader, desc=f"Epoch [{epoch:02d}/{NUM_EPOCHS:02d}] Train", leave=False)
    
    for images, targets in train_loop:
        images = list(image.to(device) for image in images)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        
        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())
        total_loss += losses.item()
        
        optimizer.zero_grad()
        losses.backward()
        optimizer.step()
        
        train_loop.set_postfix(loss=losses.item())
        
        del images, targets, loss_dict, losses
        torch.cuda.empty_cache()
        
    lr_scheduler.step()
    avg_loss = total_loss / len(train_loader)
    
    model.eval()
    metric = MeanAveragePrecision(iou_type="segm")
    
    val_loop = tqdm(val_loader, desc=f"Epoch [{epoch:02d}/{NUM_EPOCHS:02d}] Valid", leave=False)
    
    with torch.no_grad():
        for images, targets in val_loop:
            images = list(img.to(device) for img in images)
            outputs = model(images)
            
            formatted_targets = [{
                "masks": t["masks"].cpu(),
                "labels": t["labels"].cpu()
            } for t in targets]
            
            formatted_preds = [{
                "masks": (out["masks"].squeeze(1) > 0.5).to(torch.uint8).cpu(), 
                "labels": out["labels"].cpu(),
                "scores": out["scores"].cpu()
            } for out in outputs]
            
            metric.update(formatted_preds, formatted_targets)
            
            del images, outputs, formatted_preds, formatted_targets
            torch.cuda.empty_cache()

    results = metric.compute()
    
    map_overall = results['map'].item()
    map_50 = results['map_50'].item()
    map_75 = results['map_75'].item()
    mar_1 = results['mar_1'].item()
    mar_10 = results['mar_10'].item()
    mar_100 = results['mar_100'].item()
    
    precision = map_50
    recall = mar_100
    f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    
    print(f"Epoch [{epoch:02d}/{NUM_EPOCHS:02d}] | Train Loss: {avg_loss:.4f}")
    print(f"  -> Precision (mAP@0.50): {precision:.4f} | Recall (mAR@100): {recall:.4f} | F1-Score: {f1_score:.4f}")
    print(f"  -> Overall mAP (0.50:0.95): {map_overall:.4f} | mAP@0.75: {map_75:.4f}")
    print(f"  -> mAR@1: {mar_1:.4f} | mAR@10: {mar_10:.4f}\n")

    gc.collect()
    torch.cuda.empty_cache()

print("=" * 80)
print("Training completely finished!")

SAVE_PATH = "/kaggle/working/teabox_mask_rcnn.pth"
torch.save(model.state_dict(), SAVE_PATH)
print(f"\n✅ Model weights successfully saved to: {SAVE_PATH}")

/tmp/ipykernel_58/3651194646.py:17: UserWarning: Argument(s) 'value' are not valid for transform ShiftScaleRotate
  A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=15,


loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!

Starting memory-optimized training pipeline for 'TeaBox found!' for 15 epochs...


Epoch [01/15] | Train Loss: 0.2814
  -> Precision (mAP@0.50): 0.9957 | Recall (mAR@100): 0.7200 | F1-Score: 0.8357
  -> Overall mAP (0.50:0.95): 0.6745 | mAP@0.75: 0.8066
  -> mAR@1: 0.7200 | mAR@10: 0.7200



Epoch [02/15] | Train Loss: 0.2618
  -> Precision (mAP@0.50): 0.9957 | Recall (mAR@100): 0.7400 | F1-Score: 0.8490
  -> Overall mAP (0.50:0.95): 0.6944 | mAP@0.75: 0.9128
  -> mAR@1: 0.7333 | mAR@10: 0.7400



Epoch [03/15] | Train Loss: 0.2394
  -> Precision (mAP@0.50): 1.0000 | Recall (mAR@100): 0.7467 | F1-Score: 0.8550
  -> Overall mAP (0.50:0.95): 0.7089 | mAP@0.75: 0.9175
  -> mAR@1: 0.7467 | mAR@10: 0.7467



...

Epoch [11/15] | Train Loss: 0.2054
  -> Precision (mAP@0.50): 1.0000 | Recall (mAR@100): 0.7733 | F1-Score: 0.8722
  -> Overall mAP (0.50:0.95): 0.7389 | mAP@0.75: 0.9175
  -> mAR@1: 0.7733 | mAR@10: 0.7733



Epoch [12/15] | Train Loss: 0.2055
  -> Precision (mAP@0.50): 1.0000 | Recall (mAR@100): 0.7733 | F1-Score: 0.8722
  -> Overall mAP (0.50:0.95): 0.7415 | mAP@0.75: 0.9175
  -> mAR@1: 0.7733 | mAR@10: 0.7733



Epoch [13/15] | Train Loss: 0.1970
  -> Precision (mAP@0.50): 1.0000 | Recall (mAR@100): 0.7733 | F1-Score: 0.8722
  -> Overall mAP (0.50:0.95): 0.7415 | mAP@0.75: 0.9175
  -> mAR@1: 0.7733 | mAR@10: 0.7733



Epoch [14/15] | Train Loss: 0.1971
  -> Precision (mAP@0.50): 1.0000 | Recall (mAR@100): 0.7733 | F1-Score: 0.8722
  -> Overall mAP (0.50:0.95): 0.7415 | mAP@0.75: 0.9175
  -> mAR@1: 0.7733 | mAR@10: 0.7733



Epoch [15/15] | Train Loss: 0.2075
  -> Precision (mAP@0.50): 1.0000 | Recall (mAR@100): 0.7733 | F1-Score: 0.8722
  -> Overall mAP (0.50:0.95): 0.7415 | mAP@0.75: 0.9175
  -> mAR@1: 0.7733 | mAR@10: 0.7733

Training completely finished!

✅ Model weights successfully saved to: /kaggle/working/teabox_mask_rcnn.pth


In [5]:
import os
import zipfile
from IPython.display import FileLink, display

# File paths
model_path = "/kaggle/working/teabox_mask_rcnn.pth"
zip_path = "teabox_mask_rcnn.zip"

if os.path.exists(model_path):
    print("Zipping the model file... please wait.")
    
    # Compress the file
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        zipf.write(model_path, arcname=os.path.basename(model_path))
        
    print("✅ Zipping complete!")
    print("Click the link below to download your model:")
    
    # Generate the clickable link (Kaggle looks in /kaggle/working/ by default)
    display(FileLink(zip_path))
else:
    print(f"❌ Error: Could not find the file at {model_path}.")
    print("Make sure the training cell finished completely and saved the file.")

Zipping the model file... please wait.
✅ Zipping complete!
Click the link below to download your model:


/kaggle/working/teabox_mask_rcnn.zip